# 딥러닝심화 과제 7 - 기말과제 제안서
## Consistency Models를 활용한 반도체 웨이퍼 결함 패턴 생성
#### 컴퓨터공학부 인공지능공학과 20231049 정우제
<hr>

## 목차
<hr>

1. [Consistency Models](#1.Consistency-Models)
2. [모델 선택 이유 및 데이터](#2.모델-선택-이유-및-데이터)
3. [생성 목표 및 평가](#3.생성-목표-및-평가)
4. [과제 진행 계획](#4.과제-진행-계획)
5. [참고 자료](#5.참고-자료)

## 1. Consistency Models
<hr>

### 1.1 논문 정보

**제목:** Consistency Models  
**저자:** Yang Song, Prafulla Dhariwal, Mark Chen, Ilya Sutskever (OpenAI)  
**학회:** ICML 2023  
**arXiv:** https://arxiv.org/abs/2303.01469  

### 1.2 연구 배경
1. Generative Model의 변화(GAN -> Diffusion)
   * GAN의 고질적인 문제였던 Mode Collapse나 Generator와 Discriminator 간의 불안정한 학습 문제가 없다.
   * Autoregressive Model처럼 아키텍처에 제약이 있지 않아 다양한 데이터 모달리티에 적용하기 쉽다.

![image.png](./HW07_img/diffusion.png)

| 모델 | 장점 | 단점 |
|------|---|---|
| GAN | 빠른 샘플링 속도, 고해상도 생성 | 불안정한 학습, Mode Collapse, 까다로운 하이퍼파라미터 튜닝|
| VAE | 탄탄한 수학적 기반, 안정적인 학습 | 생성 결과물의 선명도가 다소 떨어짐 (Blurry samples)
| Diffusion | 최고 수준의 샘플 품질, 안정적인 학습, 다양한 모달리티 확장성	|매우 느린 샘플링 속도 (High Inference Cost)

2. Diffusion Model의 한계
    
    ![image-2.png](./HW07_img/DDPM_forward.png)
     * Forward Process - data에 가우시안 노이즈를 추가하는 과정
$$
q(x_t|x_{t-1})\sim\mathcal{N}(\sqrt{1-\beta_t}x_{t-1}, ~ \beta_t I)
$$
   * beta_t는 매우 작은 noise schedule 파라미터
  * 각 step마다 미세한 노이즈가 누적되어 최종적으로 순수 가우시안 분포에 도달
  
    ![image-4.png](./HW07_img/DDPM_reverse.png)
     * Reverse Process

$$ p_\theta(x_{t-1} | x_t) \approx \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t)) $$
   * 가정 자체가 각 스텝 간의 노이즈 차이가 매우 작아야한다.
   * 한번에 큰 변화를 주면 Gaussian 근사가 깨지기 때문에 조금씩 노이즈를 걷어내야한다.
   * 이러한 근본적인 속도 제약을 해결하기 위해 Score-SDE 관점의 재해석이 필요했다.*
1. Score-SDE
    * 기존 Diffusion Model은 noise가 생기는 과정을 discrete 단계로 정의를 했다.
    $$dx = f(x, t)dt + g(t)dw$$
      * f(x,t) : 데이터를 천천히 0으로 수렴하게 함.
      * Diffusion coefficient : 노이즈의 크기 조절
      * Standard Wiener process : 무작위 노이즈

*   위의 식을 통해 시간이 흐를수록 x가 f방향으로 조금씩 이동하면서, g만큼의 무작위 노이즈가 섞이게 된다.
    $$dx = [f(x, t) - g(t)^2 \nabla_x \log p_t(x)] dt + g(t) d\bar{w}$$
    * 위의 식이 Reverse Process과정
    * $\nabla_x \log p_t(x)$ : Score Function. 현재 노이즈 상태에서 진짜 데이터가 있는 곳을 가리키는 방향이다.
    * 생성 모델의 목표는 노이즈를 걷어내려면 어느 방향으로 가야하는지를 배운다.
    * 신경망 $s_\theta(x, t)$가 이 Score Function을 근사하도록 학습함.
   
4. Probability Flow ODE(PF ODE)
* Reverse SDE와 동일한 확률 분포를 가지지만, 노이즈 항이 없는 결정론적 방정식이다.
$$dx = \left[ f(x, t) - \frac{1}{2}g(t)^2 \nabla_x \log p_t(x) \right] dt$$
* Deterministic : 랜덤 요소가 사라져서 초기 노이즈에서 데이터까지 가는 경로가 하나로 고정된다.
* Smooth Trajectory : 부드러운 곡선을 그리며 노이즈에서 데이터로 이동한다.
* 동일한 식을 통해 노이즈 -> 데이터
* 동일한 식을 통해 0에서 T로 T에서 0으로 간다.

    ![image-4.png](./HW07_img/sde.png) 


### 1.3 Consistency Models의 핵심 아이디어
* PF ODE는 노이즈에서 데이터로 가는 결정론적 경로가 존재함을 보여줬다.
* 기존 Diffusion Model에서는 이 경로를 미분방적식 풀듯이 한 스텝씩 따라가야 했기에 속도가 느렸다.
* Consistency Model은 PF ODE 궤적의 기하학적 특성을 활용해서 반복적인 연산 없이 한번에 시작점으로 점프하는 것을 목표로 한다.
  
1. Consistency Function의 정의
* PF ODE의 궤적을 $\{x_t\}_{t \in [\epsilon, T]}$라고 할 때, 궤적 위의 임의의 점 $(x_t, t)$를 입력받아 궤적의 시작점 $x_\epsilon$으로 매핑하는 함수 $f$를 다음과 같이 정의한다.
$$f : (x_t, t) \mapsto x_\epsilon$$
  * Self-Consistency Property : 동일한 PF ODE 궤적 위에 있는 점들이라면 어떤 시점 $t$에서 출발하든 결과는 항상 같아야한다. 
  * 이를 Self-Consistency(자기 일관성)라고 하며 모델 학습의 핵심 목적함수가 된다.
$$f(x_t, t) = f(x_{t'}, t') \quad \forall t, t' \in [\epsilon, T]$$

![image-5.png](./HW07_img/consistency_methods.png)

2. Parmeterization & Boundary Condition
   * 모델이 $t=\epsilon$(노이즈가 없는 최소 시점)에서 입력 $x$를 그대로 출력하도록 경계 조건을 강제하는 기법이다.
   * Skip Connection을 이용한 구조화
$$f_\theta(x, t) = c_{skip}(t)x + c_{out}(t)F_\theta(x, t)$$
   * $t=\epsilon$일 때 $c_{skip}(\epsilon) = 1$, $c_{out}(\epsilon) = 0$이 되도록 계수 함수를 설정한다.
   * 이 경우 $f_\theta(x, \epsilon) = 1 \cdot x + 0 \cdot F_\theta(x, \epsilon) = x$가 되어 신경망 $F_\theta$의 학습 상태와 무관하게 입력이 곧 출력이 됨을 수학적으로 보장한다.

3. Sampling 방식의 차이
   * 기존 Diffusion: 순차적 denoising이 필요하다.
     * $x_T \rightarrow x_{T-1} \rightarrow \cdots \rightarrow x_1 \rightarrow x_0$
     * 각 단계마다 모델 호출
   
   * Consistency Model:
     * 1-step Generation 
       * $x_T \xrightarrow{f_\theta} x_\epsilon$ (단 1회 모델 호출)
       * 궤적의 self-consistency를 학습했기 때문에 가능하다.
     
     * Multi-step Refinement (선택적)
       * 더 높은 품질이 필요한 경우: $x_T \rightarrow x_{t_1} \rightarrow x_{t_2} \rightarrow x_\epsilon$
       * 중간 시점을 거치면서 점진적으로 정제한다.
       * 여전히 기존 방식보다 훨씬 적은 step 사용한다.

![image-6.png](./HW07_img/multistep.png)

### 1.4 모델의 학습 방법
1. Consistency Distillation(CD)
   * 이미지에 forward diffusion을 통해 노이즈를 추가한다.
   * Xt로부터 동일한 PF ODE상에 존재하는 Xt-1를 획득한다.
   * Self consistency를 만족하기 위해 F0(Xt,t)와 F0(xt-1,t-1)의 차이를 최소화하는 방향으로 학습한다.
   * 이미 학습된 Diffusion models가 있다는 것을 전제 하에 진행하는 과정이다.

![image-7.png](HW07_img\CD.png)
![image--.png](HW07_img\ConsistencyDistllation.png)


2. Consistency Traing(CT)
   * 사전에 학습된 diffusion model없이 consistency model 학습 가능하다.
   * 단독으로 학습이 가능하기 때문에 새로운 형태의 생성모델로 볼 수 있다
   * 시간 간격 $\Delta t$를 0으로 보낼수록(Time Step $N \to \infty$), 수학적으로 CT의 목적함수가 CD의 목적함수와 동일해짐을 증명하였다.
     * 스텝을 아주 잘게 쪼개서 학습하면 Teacher 모델이 없어도 Teacher가 있는 것과 똑같은 효과를 낼 수 있다.
     * Monet Carlo Estimate
$$ \nabla \log p_t(x_t) \approx -\frac{x_t - x}{\sigma_t^2} $$

![image-8.png](HW07_img\CT.png)
![image---.png](HW07_img\ConsistencyTraining.png)


   * 노이즈를 추가하는 과정으로 Xt,Xt-1를 만들어서 Consistency Model로 넣어서 Diff를 최소화 하는 방향으로 학습한다.



### 1.5 논문 결과
* 논문에서는 CIFAR-10, ImageNet 64x64, LSUN Bedroom/Cat 등의 데이터셋을 활용하여 Consistency Models(CM)의 성능을 검증했다.
  
![image-8.png](HW07_img\experiment.png)

  * Metric function을 바꾸거나 step의 수를 바꿈에 따라서 달라지는 결과를 보여준다.
  * CD의 경우 적은 수의 step임에도 불구하고 좋은 결과인 것을 볼 수 있다.


![image-8.png](HW07_img\result1.png)
  * NFE는 forward pass 중에 전체 모델 파라미터가 몇 번이나 계산되었는지를 의미한다.
  * diffusion models는 하나의 모델을 수없이 반복하기 때문에 숫자가 매우 높다.
  * CT, CD는 1~2번정도 도는 것을 확인 할 수 있다.
 
![image-8.png](HW07_img\result2.png)
  * diffusion models처럼 색칠, upsampling, 빈 공간도 채워주는 것을 볼 수 있다.   

## 2. 모델 선택 이유 및 데이터
<hr>

### 2.1 반도체 웨이퍼 결함 데이터셋
* 데이터셋 이름 : WM-811K
* 데이터셋 주소 : https://www.kaggle.com/datasets/qingyi/wm811k-wafer-map
* 데이터 구성 : 0(배경), 1(정상 칩), 2(불량 칩)
* 데이터구성 정상데이터 : 특정 불량 데이터 = 8:2

![image-81.png](HW07_img\8_category_pattern.png)

* 공정 장비의 이상이나 환경적 요인으로 발생하는 특정한 공간적 패턴으로 이상유무를 판단한다.

*    **Center:** 중앙 집중형 불량
*   **Donut:** 도넛 모양의 원형 띠
*   **Edge-Loc:** 가장자리 특정 부위 불량
*   **Edge-Ring:** 가장자리 링 형태
*   **Loc:** 특정 지역 군집(Cluster)
*   **Random:** 무작위 산재
*   **Scratch:** 긁힌 선형(Line) 불량
*   **Near-full:** 웨이퍼 전체 불량
*   **None:** 패턴 없음 (정상)

### 2.2 반도체 산업의 요구사항
* 반도체 제조 공정은 나노미터 단위의 초정밀 기술이 집약된 분야이다.
 * 실제 수집 데이터를 보아 해결해야할 과제는 극심한 데이터 불균형 문제이다.
* 정상웨이퍼 데이터는 많지만 결함이 포함된 불량 데이터는 매우 희소하게 발생하는 것을 알 수 있다.
* 딥러닝 모델로 결함을 정확히 탐지할려면 다양한 유형의 불량 패턴을 충분히 학습해야하므로 부족한 결함 데이터를 효과적으로 증강할 수 있어야한다.

### 2.3 기존 생성 모델의 한계
* 웨이퍼 데이터 증강을 위해 기존에 연구되던 생성 모델들은 반도체 산업의 요구사항을 충족시키기에 구조적인 한계를 가지고 있다. 
* 특히 뛰어난 품질로 주목받은 Diffusion Model은 생성 속도 면에서 치명적인 단점을 보인다 Diffusion Model의 Forward Process는 다음 식과 같이 데이터에 미세한 가우시안 노이즈를 점진적으로 주입하는 과정이다.
$$q(x_t|x_{t-1})\sim\mathcal{N}(\sqrt{1-\beta_t}x_{t-1}, ~ \beta_t I)$$
  
* 여기서 $\beta_t$는 매우 작은 값으로 설정되는데 이는 노이즈가 급격하게 변하지 않고 서서히 누적되도록 하기 위함이다.
* 문제는 이미지를 복원하는 Reverse Process에서 발생한다.
$$ p_\theta(x_{t-1} | x_t) \approx \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t)) $$
* 이 역확산 과정은 조건부 확률분포를 가우시안 분포로 근사하여 이전 단계의 이미지를 예측한다. 
* 이러한 가우시안 근사가 수학적으로 성립하기 위해서는 각 스텝 간의 노이즈 차이가 매우 작아야 한다는 전제 조건이 필수적이다.

### 2.4 Consistency Models 선택 이유
$$dx = \left[ f(x, t) - \frac{1}{2}g(t)^2 \nabla_x \log p_t(x) \right] dt$$

* PF ODE는 무작위 노이즈 항이 없는 결정론적 방정식으로 노이즈 상태에서 데이터 상태로 이어지는 하나의 부드러운 궤적인 Smooth Trajectory를 형성한다.

* Consistency Models는 바로 이 성질을 이용하여 궤적 위의 임의의 점을 원본 데이터 $x_0$로 직접 매핑하는 함수를 학습한다.
* 기존 모델들이 $T$에서 $T-1$을 거쳐 $0$까지 순차적으로 스텝을 밟아야 했던 것과 달리 Consistency Models는 노이즈가 섞인 상태에서 단 한 번의 단계 혹은 아주 적은 단계만으로 원본 데이터를 예측해낼 수 있다.

* 이로인해 대량의 웨이퍼 결함 데이터를 확보해야하는 반도체 공정에 꼭 필요한 모델이라고 생각이들어 선택했다.


## 3. 생성 목표 및 평가
<hr>

### 3.1 생성 목표
* 웨이퍼불량 데이터를 생성하는 것이다.
* 실제 불량 데이터와 구분이 어려울 만큼 정교한 가상 데이터를 대량으로 생산하여 데이터 불균형 문제를 해결한다.
* 기존 생성 모델 대비 속도가 빠른 Consistency Model를 활용해서 빠르게 데이터를 생성한다.

### 3.2 평가 방법
(1) 이미지 품질(FID)

* FID는 실제 데이터 분포와 생성 데이터 분포 사이의 거리를 측정하는 지표로 값이 낮을수록 실제 이미지와 유사함을 의미한다.
* Consistency Model이 적은 스텝으로도 결함 패턴을 만들어내는지 검증한다.
  
(2) 생성 속도(NFE)

* 기존 Diffusion Model이 수백 번의 연산을 필요로 하는 것과 달리 Consistency Model은 단 1회 또는 2회의 연산만으로 이미지를 생성할 수 있음을 수치로 보여준다.

(3) 시각적 검증

* Center, Donut, Scratch 등 주요 결함 패턴의 형태가 뭉개지지 않고 명확하게 구현되었는지 확인한다.

## 4. 과제 진행 계획
<hr>
(1) 데이터 수집 및 전처리

* WM-811K 데이터셋 확보
* 불량 유형별 분류 및 이미지 리사이징

(2) 모델 구현 및 학습

* Consistencty Models 기반의 생성 모델 구현
* 불량 유형별 조건부 생성 학습 진행
  
(3) 성능 평가

* 생성된 이미지의 FID 및 NFE 측정 
* 기존 Diffusion Model과의 비교를 통한 성능검증

## 5. 참고 자료
<hr>

- [Consistency Models 설명 – thecho7 블로그](https://thecho7.tistory.com/entry/%EB%85%BC%EB%AC%B8-%EB%A6%AC%EB%B7%B0-Consistency-Models-%EC%84%A4%EB%AA%85#google_vignette)
- [Consistency Models 리뷰 – kimjy99 블로그](https://kimjy99.github.io/%EB%85%BC%EB%AC%B8%EB%A6%AC%EB%B7%B0/consistency-model/)
- [DDPM 논문](https://arxiv.org/pdf/2006.11239)
- [ODE 설명](https://m.blog.naver.com/sw4r/221916036607)
- [Consistency model 논문 리뷰](https://www.youtube.com/watch?v=LGMZSqxx4Tc&t=1188s)